# AlphaMoo v4.1 — Kaggle Submission

**ARC Prize 2026 — ARC-AGI-3**

Two modes:
1. **Save Version** (no env var): Creates dummy `submission.parquet` so notebook validates
2. **Competition Rerun** (`KAGGLE_IS_COMPETITION_RERUN`): Runs AlphaMoo agent against hidden games

In [ ]:
!pip install --no-index --find-links /kaggle/input/competitions/arc-prize-2026-arc-agi-3/arc_agi_3_wheels arc-agi python-dotenv

In [ ]:
%%writefile /kaggle/working/my_agent.py
"""AlphaMoo v4.1 — ARC-AGI-3 Agent."""
import logging, random, time
from collections import deque
from dataclasses import dataclass, field
from typing import Dict, List, Optional, Set, Tuple
import numpy as np
from scipy import ndimage
from agents.agent import Agent
from arcengine import FrameData, GameAction, GameState
logger = logging.getLogger(__name__)
GRID_SIZE = 64
MAX_COLOR = 15
MIN_OBJECT_SIZE = 2
ACTION_DISPLACEMENTS = {1: (0, -1), 2: (0, 1), 3: (-1, 0), 4: (1, 0)}
@dataclass
class GameObject:
    id: str
    color: int
    cells: list
    bounding_box: tuple
    topology: str
    shape_hash: str
def detect_background_color(grid):
    from collections import Counter
    counts = Counter(grid.flatten())
    total = sum(counts.values())
    if counts.get(0, 0) > 0.5 * total:
        return 0
    return counts.most_common(1)[0][0]
def extract_objects(grid, background_color=0, min_size=MIN_OBJECT_SIZE):
    objects = []
    obj_counter = 0
    structure = np.array([[1,1,1],[1,1,1],[1,1,1]])
    for color in range(MAX_COLOR + 1):
        if color == background_color:
            continue
        mask = (grid == color)
        if not mask.any():
            continue
        labeled, n = ndimage.label(mask, structure=structure)
        for label_id in range(1, n + 1):
            comp_mask = labeled == label_id
            n_cells = int(comp_mask.sum())
            if n_cells < min_size:
                continue
            ys, xs = np.where(comp_mask)
            cells = list(zip(xs.tolist(), ys.tolist()))
            bbox = (int(xs.min()), int(ys.min()), int(xs.max()), int(ys.max()))
            obj_counter += 1
            objects.append(GameObject(
                id=f'obj_{obj_counter:03d}',
                color=int(color),
                cells=cells,
                bounding_box=bbox,
                topology='solid',
                shape_hash=str(hash(tuple(sorted(cells))) % (10**16)),
            ))
    return objects
class AgentStateTracker:
    def __init__(self):
        self.position = (32, 32)
        self.color = []
        self.shape = ''
        self.click_cursor_mode = False
        self.last_click_position = None
        self.prev_grid = None
    def update(self, grid, action_id, available_actions, click_coords=None):
        if self.prev_grid is None:
            self.prev_grid = grid.copy()
            return self.position
        if action_id in ACTION_DISPLACEMENTS:
            dx, dy = ACTION_DISPLACEMENTS[action_id]
            x, y = self.position
            self.position = (max(0, min(GRID_SIZE-1, x+dx)), max(0, min(GRID_SIZE-1, y+dy)))
        if action_id == 6 and click_coords is not None:
            self.last_click_position = click_coords
            self.click_cursor_mode = True
        self.prev_grid = grid.copy()
        return self.position
class ExperimentPlanner:
    def __init__(self):
        self.visited = set()
        self.rng = random.Random(42)
        self.recent_actions = deque(maxlen=10)
    def select_action(self, available_actions, agent_pos, objects):
        untried = [a for a in available_actions if a not in self.recent_actions]
        if untried and self.rng.random() < 0.3:
            action_id = self.rng.choice(untried)
        elif objects and any(a in available_actions for a in [1,2,3,4]):
            largest = max(objects, key=lambda o: len(o.cells))
            dx = largest.bounding_box[0] - agent_pos[0]
            dy = largest.bounding_box[1] - agent_pos[1]
            if abs(dx) > abs(dy):
                action_id = 4 if dx > 0 else 3
            else:
                action_id = 2 if dy > 0 else 1
            if action_id not in available_actions:
                action_id = self.rng.choice(available_actions)
        else:
            action_id = self.rng.choice(available_actions)
        self.recent_actions.append(action_id)
        return action_id
class MyAgent(Agent):
    def __init__(self, game_id: str = '', **kwargs):
        super().__init__(game_id=game_id, **kwargs)
        self.tracker = AgentStateTracker()
        self.planner = ExperimentPlanner()
        self.prev_grid = None
        self.step_count = 0
        self.game_id = game_id
        logger.info(f'AlphaMoo agent initialized for game {game_id}')
    def is_done(self, frames, latest_frame) -> bool:
        return latest_frame.state in (GameState.WIN, GameState.GAME_OVER)
    def choose_action(self, frames, latest_frame) -> GameAction:
        self.step_count += 1
        grid = np.array(latest_frame.frame, dtype=np.int8)
        available_actions = latest_frame.available_actions
        bg = detect_background_color(grid)
        objects = extract_objects(grid, background_color=bg)
        agent_pos = self.tracker.update(
            grid,
            latest_frame.action_input.id if latest_frame.action_input else 0,
            available_actions,
        )
        action_id = self.planner.select_action(available_actions, agent_pos, objects)
        if action_id == GameAction.ACTION6:
            if objects:
                largest = max(objects, key=lambda o: len(o.cells))
                xs = [c[0] for c in largest.cells]
                ys = [c[1] for c in largest.cells]
                click_x = sum(xs) // len(xs)
                click_y = sum(ys) // len(ys)
            else:
                click_x, click_y = 32, 32
            self.prev_grid = grid.copy()
            return GameAction.ACTION6, {'x': click_x, 'y': click_y}
        self.prev_grid = grid.copy()
        return GameAction(action_id), None

In [ ]:
import os
if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    !curl --fail --retry 999 --retry-all-errors --retry-delay 5 --retry-max-time 600 http://gateway:8001/api/games
    !cp -r /kaggle/input/competitions/arc-prize-2026-arc-agi-3/ARC-AGI-3-Agents /kaggle/working/ARC-AGI-3-Agents
    !cp /kaggle/working/my_agent.py /kaggle/working/ARC-AGI-3-Agents/agents/templates/my_agent.py
    init_content = '''from typing import Type\nfrom dotenv import load_dotenv\nfrom .agent import Agent, Playback\nfrom .templates.my_agent import MyAgent\n\nload_dotenv()\n\ndef get_agent() -> Type[Agent]:\n    return MyAgent\n'''
    with open('/kaggle/working/ARC-AGI-3-Agents/agents/__init__.py', 'w') as f:
        f.write(init_content)
    import subprocess
    result = subprocess.run(
        ['python', '-m', 'agents', '--competition'],
        cwd='/kaggle/working/ARC-AGI-3-Agents',
        capture_output=True, text=True, timeout=32000
    )
    print('STDOUT:', result.stdout[-2000:])
    print('STDERR:', result.stderr[-2000:])
    print('Competition run complete.')
else:
    print('Not a competition rerun — creating dummy submission.parquet')

In [ ]:
import os
if not os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    import pandas as pd
    submission = pd.DataFrame(
        data=[['1_0', '1', True, 1]],
        columns=['row_id', 'game_id', 'end_of_game', 'score']
    )
    submission.to_parquet('/kaggle/working/submission.parquet', index=False)
    print('Created dummy submission.parquet')
    print(submission)
else:
    if os.path.exists('/kaggle/working/submission.parquet'):
        import pandas as pd
        df = pd.read_parquet('/kaggle/working/submission.parquet')
        print(f'submission.parquet exists: {len(df)} rows')
        print(df.head())
    else:
        print('WARNING: submission.parquet not found! Creating fallback.')
        import pandas as pd
        submission = pd.DataFrame(
            data=[['1_0', '1', True, 0]],
            columns=['row_id', 'game_id', 'end_of_game', 'score']
        )
        submission.to_parquet('/kaggle/working/submission.parquet', index=False)
        print('Created fallback submission.parquet')